In [1]:
import requests
import pandas as pd
import datetime
import time
import os
from tqdm import tqdm

# ===============================================================
# CONFIGURATION
# ===============================================================
# The exact date you requested
START_DATE = "2022-11-29"
# Today's date (Includes everything up to right now)
END_DATE = datetime.datetime.today().strftime('%Y-%m-%d')
OUTPUT_FILE = "KOSPI_50_Daily_Stock_Data.parquet"

# KOSPI 50 Tickers (Yahoo Finance format uses .KS)
kospi_data = [
    {"Company": "삼성전자", "Ticker": "005930.KS"},
    {"Company": "삼성물산", "Ticker": "028260.KS"},
    {"Company": "삼성생명", "Ticker": "032830.KS"},
    {"Company": "삼성화재", "Ticker": "000810.KS"},
    {"Company": "삼성SDI", "Ticker": "006400.KS"},
    {"Company": "삼성바이오로직스", "Ticker": "207940.KS"},
    {"Company": "삼성중공업", "Ticker": "010140.KS"},
    {"Company": "SK하이닉스", "Ticker": "000660.KS"},
    {"Company": "SK", "Ticker": "034730.KS"},
    {"Company": "SK이노베이션", "Ticker": "096770.KS"},
    {"Company": "SK텔레콤", "Ticker": "017670.KS"},
    {"Company": "현대차", "Ticker": "005380.KS"},
    {"Company": "기아", "Ticker": "000270.KS"},
    {"Company": "현대모비스", "Ticker": "012330.KS"},
    {"Company": "LG", "Ticker": "003550.KS"},
    {"Company": "LG전자", "Ticker": "066570.KS"},
    {"Company": "LG에너지솔루션", "Ticker": "373220.KS"},
    {"Company": "LG화학", "Ticker": "051910.KS"},
    {"Company": "LG생활건강", "Ticker": "051900.KS"},
    {"Company": "롯데케미칼", "Ticker": "011170.KS"},
    {"Company": "롯데쇼핑", "Ticker": "023530.KS"},
    {"Company": "한화솔루션", "Ticker": "009830.KS"},
    {"Company": "GS", "Ticker": "078930.KS"},
    {"Company": "HD현대", "Ticker": "267250.KS"},
    {"Company": "HD현대중공업", "Ticker": "329180.KS"},
    {"Company": "CJ제일제당", "Ticker": "097950.KS"},
    {"Company": "두산에너빌리티", "Ticker": "034020.KS"},
    {"Company": "대한항공", "Ticker": "003490.KS"},
    {"Company": "네이버", "Ticker": "035420.KS"},
    {"Company": "카카오", "Ticker": "035720.KS"},
    {"Company": "셀트리온", "Ticker": "068270.KS"},
    {"Company": "엔씨소프트", "Ticker": "036570.KS"},
    {"Company": "넷마블", "Ticker": "251270.KS"},
    {"Company": "크래프톤", "Ticker": "259960.KS"},
    {"Company": "하이브", "Ticker": "352820.KS"},
    {"Company": "에코프로", "Ticker": "086520.KQ"}, # KQ for KOSDAQ
    {"Company": "금양", "Ticker": "001570.KS"},
    {"Company": "DB손해보험", "Ticker": "005830.KS"},
    {"Company": "아모레퍼시픽", "Ticker": "090430.KS"},
    {"Company": "POSCO홀딩스", "Ticker": "005490.KS"},
    {"Company": "포스코퓨처엠", "Ticker": "003670.KS"},
    {"Company": "KT", "Ticker": "030200.KS"},
    {"Company": "KT&G", "Ticker": "033780.KS"},
    {"Company": "HMM", "Ticker": "011200.KS"},
    {"Company": "S-Oil", "Ticker": "010950.KS"},
    {"Company": "한국전력", "Ticker": "015760.KS"},
    {"Company": "KB금융", "Ticker": "105560.KS"},
    {"Company": "신한지주", "Ticker": "055550.KS"},
    {"Company": "하나금융지주", "Ticker": "086790.KS"},
    {"Company": "기업은행", "Ticker": "024110.KS"}
]

# ===============================================================
# DIRECT API SCRAPER (No Pagination = No Limits)
# ===============================================================
def get_full_history(ticker, start_date, end_date):
    # 1. Calculate Timestamps
    #    We subtract 1 day from start and add 1 day to end to be safe with timezones
    start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d") - datetime.timedelta(days=1)
    end_dt = datetime.datetime.strptime(end_date, "%Y-%m-%d") + datetime.timedelta(days=1)

    start_ts = int(time.mktime(start_dt.timetuple()))
    end_ts = int(time.mktime(end_dt.timetuple()))

    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"

    # 2. Request EVERYTHING in one go
    params = {
        "symbol": ticker,
        "period1": start_ts,
        "period2": end_ts,
        "interval": "1d",          # Daily data
        "includePrePost": "false", # Regular market hours only
        "events": "div|split"      # Include splits/dividends just in case
    }

    # Fake Browser Header
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        # Retry loop in case of network blip
        for attempt in range(3):
            try:
                response = requests.get(url, headers=headers, params=params, timeout=10)
                if response.status_code == 200:
                    break
            except:
                time.sleep(2)

        if response.status_code != 200:
            print(f"FAILED to fetch {ticker}: HTTP {response.status_code}")
            return None

        data = response.json()

        # 3. Parse JSON
        if "chart" not in data or "result" not in data["chart"] or not data["chart"]["result"]:
            print(f"NO DATA returned for {ticker}")
            return None

        result = data["chart"]["result"][0]
        timestamps = result["timestamp"]
        quote = result["indicators"]["quote"][0]

        # Safe extraction of Adj Close
        adj_close = quote["close"] # Default
        if "adjclose" in result["indicators"]:
             adj_data = result["indicators"]["adjclose"][0]
             if "adjclose" in adj_data:
                 adj_close = adj_data["adjclose"]

        # 4. Create DataFrame
        df = pd.DataFrame({
            "Date": pd.to_datetime(timestamps, unit="s"),
            "Close": quote["close"],
            "Adj Close": adj_close,
            "Volume": quote["volume"]
        })

        # Drop empty days (holidays)
        df.dropna(inplace=True)
        df["Date"] = df["Date"].dt.date

        return df

    except Exception as e:
        print(f"Error parsing {ticker}: {e}")
        return None

# ===============================================================
# MAIN EXECUTION
# ===============================================================
if __name__ == "__main__":
    all_data = []

    # -----------------------------------------------------------
    # STEP 1: Get the KOSPI Index (Market Benchmark)
    # -----------------------------------------------------------
    print("Downloading KOSPI Index (^KS11)...")
    df_kospi = get_full_history("^KS11", START_DATE, END_DATE)

    if df_kospi is None or df_kospi.empty:
        print("CRITICAL ERROR: Cannot get KOSPI data. Exiting.")
        exit()

    # Prepare KOSPI for merging (Rename cols)
    df_kospi = df_kospi[["Date", "Close", "Adj Close"]].rename(
        columns={"Close": "KOSPI_Close", "Adj Close": "KOSPI_Adj_Close"}
    )

    print(f" -> Got {len(df_kospi)} days of KOSPI data.")

    # -----------------------------------------------------------
    # STEP 2: Get All 50 Companies
    # -----------------------------------------------------------
    print(f"\nDownloading {len(kospi_data)} Companies...")

    for item in tqdm(kospi_data):
        ticker = item["Ticker"]
        company = item["Company"]

        # Fetch
        df = get_full_history(ticker, START_DATE, END_DATE)

        if df is not None and not df.empty:
            # Tag the data
            df["Company"] = company
            df["Ticker"] = ticker

            # MERGE: Attach KOSPI prices to this company's rows
            # This ensures every day has both Stock Price AND Market Price
            df_merged = pd.merge(df, df_kospi, on="Date", how="left")

            all_data.append(df_merged)

        time.sleep(1) # Be nice to the API

    # -----------------------------------------------------------
    # STEP 3: Save Final File
    # -----------------------------------------------------------
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        # Reorder for cleanliness
        cols = ["Date", "Company", "Ticker", "Close", "Adj Close", "KOSPI_Close", "KOSPI_Adj_Close", "Volume"]
        final_df = final_df[cols]

        final_df.to_parquet(OUTPUT_FILE, index=False)

        print("\n" + "="*40)
        print("DOWNLOAD COMPLETE")
        print(f"Saved to: {OUTPUT_FILE}")
        print(f"Total Rows: {len(final_df)}")
        print(f"Date Range: {final_df['Date'].min()} to {final_df['Date'].max()}")
        print("="*40)
        print(final_df.head())
    else:
        print("Something went wrong. No data collected.")

 -> Got 747 days of KOSPI data.



100%|██████████| 50/50 [00:57<00:00,  1.15s/it]


DOWNLOAD COMPLETE
Saved to: KOSPI_50_Daily_Stock_Data.parquet
Total Rows: 37289
Date Range: 2022-11-28 to 2025-12-17
         Date Company     Ticker    Close     Adj Close  KOSPI_Close  \
0  2022-11-28    삼성전자  005930.KS  60100.0  56329.601562  2408.270020   
1  2022-11-29    삼성전자  005930.KS  60600.0  56798.226562  2433.389893   
2  2022-11-30    삼성전자  005930.KS  62200.0  58297.855469  2472.530029   
3  2022-12-01    삼성전자  005930.KS  62600.0  58672.757812  2479.840088   
4  2022-12-02    삼성전자  005930.KS  60400.0  56610.777344  2434.330078   

   KOSPI_Adj_Close      Volume  
0      2408.270020   8589032.0  
1      2433.389893   7014160.0  
2      2472.530029  19768903.0  
3      2479.840088  16631445.0  
4      2434.330078  15331184.0  
